# 02 — Scripts and files

LAMMPS runs in an in-memory filesystem (working directory `/work`) and reads
and writes it exactly like a real disk: input scripts, data files, potentials,
dumps, logs. This notebook runs input files, reads results back out, and
fetches a script over HTTP — the browser equivalents of `lmp.file(...)`
workflows on a real machine.

In [ ]:
%pip install lammps-js

## Run an input file

`lmp.file(path)` is the official API for running an input script from a file.
In the browser you can hand it the file body directly with `contents=` — it
writes the file into the wasm filesystem and runs it. This script also dumps
the final configuration to `atoms.dump`:

In [ ]:
from lammps import lammps

lmp = await lammps()
lmp.file("melt.in", contents="""
units         lj
atom_style    atomic
lattice       fcc 0.8442
region        box block 0 2 0 2 0 2
create_box    1 box
create_atoms  1 box
mass          1 1.0
velocity      all create 3.0 87287
pair_style    lj/cut 2.5
pair_coeff    1 1 1.0 1.0 2.5
fix           1 all nve
thermo        100
run           200
write_dump    all atom atoms.dump
""")

## Read files back out

Anything LAMMPS wrote can be read back — parse it with numpy, plot it, or
save it. `read_file` returns a text file from the wasm filesystem; a LAMMPS
atom dump has 9 header lines followed by one row per atom:

In [ ]:
import numpy as np

dump = lmp.read_file("atoms.dump")
print("\n".join(dump.splitlines()[:9]))

data = np.loadtxt(dump.splitlines()[9:])
print("\ndump columns (id type xs ys zs):", data.shape)
print("first atom:", data[0])

## Write files in

`write_file` puts any text (or bytes) into the wasm filesystem — this is how
you provide data files, restart files, or potential files before a run:

In [ ]:
lmp.write_file("note.txt", "any bytes or text")
print(lmp.read_file("note.txt"))

## Fetch a script from the web

Data files shipped with this site (the `data/` folder in the file browser)
are served over HTTP. `lammps.site_url` builds the URL, `pyfetch` downloads —
the same pattern works for any potential or data file on the web (e.g. GitHub
raw URLs):

In [ ]:
from pyodide.http import pyfetch
import lammps as ljs

response = await pyfetch(ljs.site_url("files/data/lj-melt.in"))
script = await response.string()
print(script[:120], "…")

lmp2 = await lammps(output=None)
lmp2.file("lj-melt.in", contents=script)
print("ran lj-melt.in:", lmp2.get_natoms(), "atoms at T =", round(lmp2.get_thermo("temp"), 3))
lmp2.close()
lmp.close()

Next: [03 — Analysis and plotting](03-analysis-and-plotting.ipynb) with numpy
and matplotlib.